# kigurumi 头壳尺寸计算器 · Colab 版

从 **演员身体数据 + 角色设定比例 + 假发留量** 反推头壳该做多大，并校验戴不戴得上。

**怎么用**：先跑「引擎装载」，之后 A / B / C 三节可反复调参重跑。

| 节 | 干什么 |
|---|---|
| **A** | 上传角色正面立绘，自动量出 头身比 `N` 与 头宽/肩宽 `r`（可跳过，手填也行） |
| **B** | 填演员实测数据 → 输出净壳 / 含发 的 长宽高、比例、可戴性校验、重量估算 |
| **C** | 扫描图：这个演员到底能做到几头身、肩要垫多宽 |

---

### 两点必须先说清楚

**A 节是几何启发式，不是识别模型。** 它靠剪影的宽度剖面找头顶/下巴/颈/肩四条线。长发盖住脖子、抱手臂、大裙摆、动态姿势都会骗过它。合成标定图上 `r` 误差约 1%，`N` 误差约 4%——误差几乎全在下巴线上，因为剪影里下巴和脖子的交界本来就是模糊的。**每次都会输出标注图，请看一眼再用结果**，四条线都能手动覆盖。

**假发留量的定义**：从该侧「头壳基底材质的外表面」，沿该方向垂直量到发丝外缘的距离。发套底网、定位胶带、发丝蓬松量全算在这一个数里；壁厚不算，它在壳外皮以内，不会重复计算。所以正面看：`含发头宽 = 净壳宽 + 2 × 侧向留量`。

> 本 notebook **不含算法副本**：它直接 `import` `kigurumi_head_calc.py` / `char_ratio_from_image.py`（见下方「引擎装载」），改脚本即改 notebook。

## 引擎装载（运行一次）

本 notebook **不再自带算法副本**，直接 `import` 仓库里的两份脚本（单一来源，改脚本即改 notebook）：

| 别名 | 文件 | 负责 |
|---|---|---|
| `khc` | `kigurumi_head_calc.py` | 头壳尺寸计算 |
| `cri` | `char_ratio_from_image.py` | 立绘比例识别（头身比 / 头宽肩宽比） |

把这两个 `.py` 和 notebook 放同一目录即可；Colab 上运行下面的 cell 会提示你上传。


In [ ]:
#@title 【0】装载引擎 { display-mode: "form" }
import os, sys, subprocess

# Colab 上装依赖（本地已装会跳过）
try:
    import google.colab  # noqa: F401
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "Pillow", "opencv-python<5.0.0", "rembg", "onnxruntime"])
except ImportError:
    pass


def _ensure_importable():
    """让 kigurumi_head_calc.py / char_ratio_from_image.py 所在目录可被 import。"""
    for d in (os.getcwd(), "SizeCalc", os.path.join(os.getcwd(), "SizeCalc")):
        if os.path.exists(os.path.join(d, "kigurumi_head_calc.py")):
            if d not in sys.path:
                sys.path.insert(0, d)
            return True
    return False


if not _ensure_importable():
    try:
        from google.colab import files
        print("请上传 kigurumi_head_calc.py 与 char_ratio_from_image.py：")
        files.upload()
        if os.getcwd() not in sys.path:
            sys.path.insert(0, os.getcwd())
    except Exception:
        raise SystemExit("找不到 kigurumi_head_calc.py / char_ratio_from_image.py，"
                         "请把它们放到 notebook 同目录（Colab 里则按提示上传）")

import kigurumi_head_calc as khc
import char_ratio_from_image as cri

# 图上标签：Colab 常没中文字体，找不到就退英文，绝不显示豆腐块
import matplotlib
import matplotlib.pyplot as plt


def _pick_cjk_font():
    import matplotlib.font_manager as fm
    for name in ("Noto Sans CJK JP", "Noto Sans CJK SC", "WenQuanYi Zen Hei",
                 "Source Han Sans SC", "PingFang SC", "SimHei"):
        try:
            if fm.findfont(name, fallback_to_default=False):
                matplotlib.rcParams["font.family"] = name
                matplotlib.rcParams["axes.unicode_minus"] = False
                return True
        except Exception:
            pass
    return False


HAS_CJK = _pick_cjk_font()


def L(zh, en):
    return zh if HAS_CJK else en


print("✅ 引擎已装载：khc / cri" + ("（中文字体可用）" if HAS_CJK else "（无中文字体，图上标签用英文）"))


## A · 上传立绘，自动量比例

带**透明背景的 PNG** 最准。没有立绘就跳过这一节，直接去 B 手填 `N` 和 `r`。

跑完对着标注图检查四条线；不对就在下面的 `chin_y` / `top_y` / `foot_y` / `head_w` / `shoulder_w`
填**原图像素坐标**覆盖（填 0 = 用自动值），重跑本 cell。

In [ ]:
#@title 【A】上传立绘并自动量比例 { display-mode: "form" }
#@markdown **图片来源**：留空则弹出上传框；也可填路径或图片 URL
image_source = "" #@param {type:"string"}
#@markdown ---
工作分辨率上限 = 1000 #@param {type:"slider", min:400, max:2000, step:100}
背景色容差 = 30 #@param {type:"slider", min:5, max:90, step:5}
肩线搜索范围_乘头高 = 0.6 #@param {type:"slider", min:0.2, max:1.5, step:0.1}
#@markdown ---
#@markdown **手动覆盖**（原图像素坐标，0 = 用自动值）
chin_y = 0 #@param {type:"integer"}
top_y = 0 #@param {type:"integer"}
foot_y = 0 #@param {type:"integer"}
head_w = 0 #@param {type:"integer"}
shoulder_w = 0 #@param {type:"integer"}

from PIL import Image

src = image_source.strip()
if not src:
    from google.colab import files
    up = files.upload()
    if not up:
        raise SystemExit("没有选择文件")
    src = list(up.keys())[0]

w, h, rgba, scale, how = cri.load_image(src, int(工作分辨率上限))
mask, mhow = cri.build_mask(w, h, rgba, 128, int(背景色容差))
ov = {k: (v / scale if v else None) for k, v in
      dict(top_y=top_y, chin_y=chin_y, foot_y=foot_y,
           head_w=head_w, shoulder_w=shoulder_w).items()}
face_rects, face_note = cri.detect_faces(w, h, rgba)
m = cri.analyse(w, h, mask, ov, float(肩线搜索范围_乘头高), face_rects, None, 1.45)
if face_note:
    m["warnings"].insert(0, face_note)

AUTO_N, AUTO_R = round(m["ratio"], 2), round(m["head_shoulder"], 3)

print("图片 %s" % src)
print("读取：%s    抠图：%s    工作分辨率 %d×%d（原图约为其 %.2f 倍）"
      % (how, mhow, w, h, scale))
print("线位(工作图 y)：头顶 %d  下巴 %d  颈最窄 %d  脚底 %d   下巴线来源：%s"
      % (m["top_y"], m["chin_y"], m["neck_y"], m["foot_y"], m["chin_src"]))
print("全身高 %d px   头高 %d px   头宽 %.0f px   肩宽 %.0f px   颈宽 %d px"
      % (m["total_px"], m["head_h_px"], m["head_w_px"], m["shoulder_w_px"], m["neck_w_px"]))
print("★ 头身比    N = %d / %d = %.2f" % (m["total_px"], m["head_h_px"], m["ratio"]))
print("★ 头宽/肩宽 r = %.0f / %.0f = %.3f"
      % (m["head_w_px"], m["shoulder_w_px"], m["head_shoulder"]))
for s in m["warnings"]:
    print("⚠ " + s)

plt.figure(figsize=(5, 10))
plt.imshow(Image.frombytes("RGBA", (w, h), bytes(cri.annotate(w, h, rgba, m))))
plt.axis("off")
plt.title(L("标注图：红线=头顶/下巴  蓝=脚底  绿=头宽  橙=肩宽",
            "annotated: verify the four lines"))
plt.show()
print("已存入 AUTO_N = %s，AUTO_R = %s，B 节可直接调用" % (AUTO_N, AUTO_R))


## B · 演员数据 → 头壳尺寸

`用A节自动值` 打勾就吃 A 的结果；不打勾就用你手填的 `头身比` / `头宽肩宽比`。

假发那几个值填 **0 表示按基准值自动派生**（系数：顶 1.0 / 侧 1.0 / 前 0.6 / 后 1.4）。
侧向的 1.0 只是没有实测时的兜底，能量到就直接填。

In [ ]:
#@title 【B】算头壳尺寸 { display-mode: "form" }
#@markdown ### 演员实测数据（cm / kg）
身高 = 168 #@param {type:"number"}
体重 = 52 #@param {type:"number"}
肩宽 = 38 #@param {type:"number"}
头宽 = 15.5 #@param {type:"number"}
头高 = 22.5 #@param {type:"number"}
#@markdown 下面两项留 0 = 按经验比例估算
头深 = 0 #@param {type:"number"}
肩峰高 = 0 #@param {type:"number"}

#@markdown ### 角色设定
用A节自动值 = True #@param {type:"boolean"}
头身比 = 6.0 #@param {type:"number"}
头宽肩宽比 = 0.65 #@param {type:"number"}
头深宽比 = 1.15 #@param {type:"slider", min:1.0, max:1.35, step:0.01}
头身比含发型 = True #@param {type:"boolean"}

#@markdown ### 假发留量（自头壳基底材质外表面量到发丝外缘，cm；0 = 自动派生）
基准留量 = 3.0 #@param {type:"number"}
顶 = 0 #@param {type:"number"}
侧_单侧 = 0 #@param {type:"number"}
前 = 0 #@param {type:"number"}
后 = 0 #@param {type:"number"}

#@markdown ### 海绵留量（贴壳内壁的衬垫**标称**厚度，cm；0 = 自动派生）
#@markdown 和下面的「空气余量」不是一回事：海绵是实体、可压缩、负责夹住头；空气余量是风扇/走线/通风的空隙
海绵基准厚度 = 2.0 #@param {type:"number"}
海绵_顶 = 0 #@param {type:"number"}
海绵_侧单侧 = 0 #@param {type:"number"}
海绵_面 = 0 #@param {type:"number"}
海绵_后 = 0 #@param {type:"number"}
海绵压缩率 = 0.3 #@param {type:"slider", min:0, max:0.8, step:0.05}
海绵密度_g_cm3 = 0.05 #@param {type:"number"}

#@markdown ### 制作参数
支撑方式 = "chin" #@param ["chin", "shoulder", "custom"]
底沿低于下巴 = 5.0 #@param {type:"number"}
增高鞋厚度 = 0 #@param {type:"number"}
壁厚 = 0.5 #@param {type:"number"}
单侧空气余量 = 1.0 #@param {type:"number"}
头顶空气余量 = 1.5 #@param {type:"number"}
面部空气余量 = 2.0 #@param {type:"number"}
后脑空气余量 = 1.5 #@param {type:"number"}
单侧肩部填充 = 0 #@param {type:"number"}
材料 = "pla" #@param ["pla", "petg", "abs", "resin", "glassfiber"]

from IPython.display import HTML, display
from html import escape as _esc   # 警告里有 "<" 之类的字符，不转义会被浏览器吞掉后半句

N = globals().get("AUTO_N") if 用A节自动值 else None
r = globals().get("AUTO_R") if 用A节自动值 else None
src_note = "取自 A 节自动测量" if (N and r) else "手填"
if not (N and r):
    N, r = 头身比, 头宽肩宽比

body = khc.Body(身高, 体重, 肩宽, 头宽, 头高, (头深 or None), (肩峰高 or None))
char = khc.Character(N, r, 头深宽比, 头身比含发型)
build = khc.Build(
    mount=支撑方式, rim_drop=底沿低于下巴, lift=增高鞋厚度,
    wig=基准留量, wig_top=(顶 or None), wig_side=(侧_单侧 or None),
    wig_front=(前 or None), wig_back=(后 or None),
    foam=海绵基准厚度, foam_top=(海绵_顶 or None), foam_side=(海绵_侧单侧 or None),
    foam_face=(海绵_面 or None), foam_back=(海绵_后 or None),
    foam_compress=海绵压缩率, foam_density=海绵密度_g_cm3,
    side_clear=单侧空气余量, top_clear=头顶空气余量,
    face_clear=面部空气余量, back_clear=后脑空气余量,
    wall=壁厚, shoulder_pad=(单侧肩部填充 or None), material=材料,
)
RESULT = khc.compute(body, char, build)
EYE_FROM_TOP_MIN, EYE_FROM_TOP_MAX = khc.EYE_FROM_TOP_MIN, khc.EYE_FROM_TOP_MAX


def _dwh(x):
    return {"D": x["深"], "W": x["宽"], "H": x["高"]}


# 把 khc 的中文结果字典映射成下面模板用的短键
R = {
    "h_mount": RESULT["人体基准"]["底沿离地"],
    "mount_desc": RESULT["人体基准"]["支撑方式"],
    "shoulder_cos": RESULT["人体基准"]["着装后肩宽"],
    "total_height": RESULT["成品总身高"],
    "net": _dwh(RESULT["净壳尺寸"]), "vis": _dwh(RESULT["含发外观尺寸"]),
    "minimum": _dwh(RESULT["可戴最小净壳"]),
    "ok": {"D": RESULT["校验"]["深"], "W": RESULT["校验"]["宽"],
           "H": RESULT["校验"]["高"], "all": RESULT["校验"]["整体可行"]},
    "chain": {"W": RESULT["尺寸链"]["宽"], "H": RESULT["尺寸链"]["高"],
              "D": RESULT["尺寸链"]["深"]},
    "wig": {"top": RESULT["假发厚度"]["顶"], "side": RESULT["假发厚度"]["侧(单侧)"]},
    "foam": {"nom_top": RESULT["海绵"]["标称_顶"], "nom_side": RESULT["海绵"]["标称_侧单侧"],
             "nom_face": RESULT["海绵"]["标称_面"], "nom_back": RESULT["海绵"]["标称_后"],
             "eff_top": RESULT["海绵"]["实占_顶"], "eff_side": RESULT["海绵"]["实占_侧单侧"],
             "eff_face": RESULT["海绵"]["实占_面"], "eff_back": RESULT["海绵"]["实占_后"],
             "compress": RESULT["海绵"]["压缩率"]},
    "eye": {"from_top": RESULT["眼位"]["演员眼睛离壳顶"],
            "ratio": RESULT["眼位"]["占净壳高比例"], "ok": RESULT["眼位"]["合格"]},
    "mass": {"material": RESULT["重量估算"]["材料"], "wall_mm": RESULT["重量估算"]["壁厚_mm"],
             "shell_kg": RESULT["重量估算"]["壳体估重_kg"],
             "foam_kg": RESULT["重量估算"]["海绵估重_kg"],
             "total_kg": RESULT["重量估算"]["合计_kg"],
             "limit_kg": RESULT["重量估算"]["建议上限_kg"]},
    "n_max": RESULT["极限"]["该演员最大可行头身比"],
    "pad_needed": RESULT["极限"]["达成当前头宽所需单侧肩部填充"],
    "extra_lift": RESULT["极限"]["达成当前头身比所需增高"],
    "notes": RESULT["说明"], "warnings": RESULT["警告"], "advice": RESULT["建议"],
}

def _rows(d, keys, unit=""):
    return "".join("<td style='text-align:right;padding:4px 14px'>%.1f%s</td>" % (d[k], unit)
                   for k in keys)


def _chain_text(R):
    """尺寸链：可戴最小净壳是怎么一段段堆出来的。"""
    names = {"W": "宽", "H": "高", "D": "深"}
    out = []
    for k in ("W", "H", "D"):
        expr = " + ".join("%s %.1f" % (nm, v) for nm, v in R["chain"][k])
        # 合计显示未取整的真值，避免和上表的最小值差 0.1
        out.append("%s：%s\n      = %.1f cm  ← 最小%s"
                   % (names[k], expr, R["minimum"][k], names[k]))
    return _esc("\n".join(out))

badge = ("<span style='background:#1f9d55;color:#fff;padding:3px 10px;border-radius:4px'>✓ 可行</span>"
         if R["ok"]["all"] else
         "<span style='background:#c53030;color:#fff;padding:3px 10px;border-radius:4px'>✗ 做不出来</span>")

html = """
<div style="font-family:-apple-system,'Segoe UI',sans-serif;line-height:1.55;max-width:900px">
<h3 style="margin:.2em 0">头壳尺寸 &nbsp; %s</h3>
<p style="color:#666;margin:.2em 0 1em">角色 %.2f 头身，头宽/肩宽 %.3f（%s）&nbsp;·&nbsp;
%s&nbsp;·&nbsp;底沿离地 %.1f cm&nbsp;·&nbsp;成品总身高 %.1f cm</p>

<table style="border-collapse:collapse;margin-bottom:1.2em">
<tr style="background:#f0f0f0"><th style="padding:6px 14px;text-align:left"></th>
<th style="padding:6px 14px">深(前后)</th><th style="padding:6px 14px">宽(左右)</th>
<th style="padding:6px 14px">高(上下)</th></tr>
<tr style="background:#fffbe6;font-weight:600"><td style="padding:4px 14px">净壳（建模用）</td>%s</tr>
<tr><td style="padding:4px 14px">含发外观（成品）</td>%s</tr>
<tr style="color:#888"><td style="padding:4px 14px">可戴最小值</td>%s</tr>
<tr style="color:#888"><td style="padding:4px 14px">余量</td>%s</tr>
</table>

<p><b>净壳比例</b> 深:宽:高 = %.3f : 1.000 : %.3f &nbsp;&nbsp;|&nbsp;&nbsp;
<b>建模包围盒</b> %.1f × %.1f × %.1f mm</p>

<h4 style="margin:1em 0 .3em">海绵留量与尺寸链</h4>
<p style="margin:.2em 0">标称厚度 顶 %.2f / 侧(单侧) %.2f / 面 %.2f / 后 %.2f cm&nbsp;&nbsp;—&nbsp;&nbsp;
压缩 %.0f%% → 实占 顶 %.2f / 侧 %.2f / 面 %.2f / 后 %.2f cm<br>
<span style="color:#888">海绵按标称厚度买，戴上后被压薄：占位算实占，重量算标称。</span></p>
<pre style="background:#f7f7f7;padding:10px 12px;border-radius:4px;font-size:12px;
line-height:1.7;overflow-x:auto;margin:.4em 0">%s</pre>

<p><b>眼位</b>：演员眼睛离壳顶 %.1f cm，占净壳高 %.1f%%（角色眼位建议 %.0f%%~%.0f%%）→ %s<br>
<b>重量</b>：%s 壁厚 %.1f mm —— 壳体 %.2f + 海绵 %.2f = <b>%.2f kg</b>
（上限 %.2f kg；涂装+假发+风扇另计 0.4~1.2 kg）<br>
<b>极限</b>：该演员最大可行 %.2f 头身；达成当前头宽需单侧肩部填充 %.1f cm%s</p>
</div>
""" % (
    badge, N, r, src_note, R["mount_desc"], R["h_mount"], R["total_height"],
    _rows(R["net"], "DWH"), _rows(R["vis"], "DWH"),
    _rows(R["minimum"], "DWH"),
    "".join("<td style='text-align:right;padding:4px 14px'>%+.1f</td>"
            % (R["net"][k]-R["minimum"][k]) for k in "DWH"),
    R["net"]["D"]/R["net"]["W"], R["net"]["H"]/R["net"]["W"],
    R["net"]["D"]*10, R["net"]["W"]*10, R["net"]["H"]*10,
    R["foam"]["nom_top"], R["foam"]["nom_side"], R["foam"]["nom_face"], R["foam"]["nom_back"],
    R["foam"]["compress"]*100,
    R["foam"]["eff_top"], R["foam"]["eff_side"], R["foam"]["eff_face"], R["foam"]["eff_back"],
    _chain_text(R),
    R["eye"]["from_top"], R["eye"]["ratio"]*100,
    EYE_FROM_TOP_MIN*100, EYE_FROM_TOP_MAX*100,
    "对得上" if R["eye"]["ok"] else "需内部调整头台高度",
    R["mass"]["material"], R["mass"]["wall_mm"], R["mass"]["shell_kg"],
    R["mass"]["foam_kg"], R["mass"]["total_kg"], R["mass"]["limit_kg"],
    R["n_max"], R["pad_needed"],
    ("；达成当前头身比还需增高 %.1f cm" % R["extra_lift"]) if R["extra_lift"] > 0 else "",
)
display(HTML(html))

for tag, items, color in (("估算说明", R["notes"], "#666"),
                          ("⚠ 警告", R["warnings"], "#b7791f"),
                          ("→ 建议", R["advice"], "#2b6cb0")):
    if items:
        display(HTML("<div style='font-family:-apple-system,sans-serif;color:%s;max-width:900px'>"
                     "<b>%s</b><ul style='margin:.3em 0'>%s</ul></div>"
                     % (color, tag, "".join("<li>%s</li>" % _esc(s) for s in items))))

## C · 可行性扫描

两张图回答两个问题：**这位演员最多能做到几头身？** **肩要垫多宽才够？**

阴影区 = 壳体小到塞不进人头的不可行区。

In [ ]:
#@title 【C】扫描图 { display-mode: "form" }
头身比扫描范围 = "3, 9" #@param {type:"string"}
头宽肩宽比扫描范围 = "0.4, 1.1" #@param {type:"string"}

import numpy as np, matplotlib.pyplot as plt
R = globals().get("R")
if R is None:
    raise SystemExit("请先运行 B 节")

wt, ws_ = R["wig"]["top"], R["wig"]["side"]
h_mount, hs_min, ws_min = R["h_mount"], R["minimum"]["H"], R["minimum"]["W"]
hair = 头身比含发型

n0, n1 = [float(x) for x in 头身比扫描范围.split(",")]
r0, r1 = [float(x) for x in 头宽肩宽比扫描范围.split(",")]
Ns = np.linspace(max(1.05, n0), n1, 300)
Rs = np.linspace(r0, r1, 300)

hs = (h_mount/(Ns-1) - wt) if hair else ((h_mount + wt)/(Ns-1))
wsv = Rs*R["shoulder_cos"] - 2*ws_

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.4))

a1.plot(Ns, hs, color="#2b6cb0", lw=2)
a1.axhline(hs_min, color="#c53030", ls="--", lw=1.4)
a1.fill_between(Ns, 0, hs_min, color="#c53030", alpha=0.10)
a1.axvline(R["n_max"], color="#1f9d55", ls=":", lw=1.4)
a1.plot([globals().get("AUTO_N") or 头身比], [R["net"]["H"]], "o", color="#f09628", ms=9, zorder=5)
a1.set_xlabel(L("头身比 N", "head-body ratio N"))
a1.set_ylabel(L("净壳高 (cm)", "net shell height (cm)"))
a1.set_title(L("最大可行 %.2f 头身（绿虚线）", "max feasible N = %.2f") % R["n_max"], fontsize=10)
a1.set_ylim(0, max(hs_min*2.2, R["net"]["H"]*1.4)); a1.grid(alpha=0.25)
a1.text(Ns[-1], hs_min, L("  可戴下限 %.1f", "  min %.1f") % hs_min,
        color="#c53030", va="bottom", ha="right", fontsize=9)

a2.plot(Rs, wsv, color="#2b6cb0", lw=2)
a2.axhline(ws_min, color="#c53030", ls="--", lw=1.4)
a2.fill_between(Rs, 0, ws_min, color="#c53030", alpha=0.10)
a2.plot([globals().get("AUTO_R") or 头宽肩宽比], [R["net"]["W"]], "o", color="#f09628", ms=9, zorder=5)
a2.set_xlabel(L("头宽 / 肩宽  r", "head/shoulder ratio r"))
a2.set_ylabel(L("净壳宽 (cm)", "net shell width (cm)"))
a2.set_title(L("当前肩宽 %.1f cm 下", "at shoulder width %.1f cm") % R["shoulder_cos"], fontsize=10)
a2.set_ylim(0, max(ws_min*2.2, R["net"]["W"]*1.4)); a2.grid(alpha=0.25)
a2.text(Rs[-1], ws_min, L("  可戴下限 %.1f", "  min %.1f") % ws_min,
        color="#c53030", va="bottom", ha="right", fontsize=9)

plt.tight_layout(); plt.show()
print("橙点 = 当前设定。落在阴影区里就是做不出来。")